## Environment Setup
Do not modify the following two cells. adh_init() details here: [README-adh_init](https://github.com/cfacorp/aao-dbx-helpers/blob/master/README-adh_init.md).

In [0]:
%run /Workspace/Repos/global/aao-dbx-helpers/.boot/adh_init

Looking in indexes: https://[REDACTED]:****@[REDACTED]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.1/98.1 MB 81.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 65.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.9/603.9 kB 48.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 88.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 89.8 MB/s eta 0:00:00
  Attempting uninstall: typing_extensions
    Found existing installation: typing_extensions 4.12.2
    Not uninstalling typing-extensions at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e31f4a8b-b240-40b7-bff9-2f5cd7911608
    Can't uninstall 'typing_extensions'. No files were found to uninstall.
  Attempting unin

In [0]:
__adh_init()

ADHObject(params=ADHOParamsObject(start_time='1789701096186', parent_run_id='001789701096186', run_id='001789701096186', workspace_directory='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/jobs/mom-capacity-ml-cc-sales-forecast', start_date_of_analysis='2017-01-01', environment='dev', task_retry_count='0', trigger_date='2026-09-18', task_key='sales-forecast', start_date='2026-09-18', global_workspace_directory='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/global_files', trigger_time='1789701096186', task_execution_count='1', workflow_name='mom-capacity-ml-cc-sales-forecast', job_id='0'), conf=ADHConfObject(default_schema='su_bae_test', default_catalog='cfaadvanalyticsdev_primary', default_schema_qualified='cfaadvanalyticsdev_primary.su_bae_test', repo_unique_id='mom-capacity-ml-cc', default_dx_catalog='dx_test', nb_path='/Workspace/Users/su.bae@cfacorp.com/dbx-jobs-cfaadvanalytics-mom-capacity-ml-cc/jobs/mom-capacity-ml-cc-sales-forecast/sales-forecast', storage_base='s3://cfaadvanalyticsdev-us-east-1-databricks-primary-data/user_test/su_bae/', redshift_tmpdir='s3://cfaadvanalyticsdev-us-east-1-databricks-primary-data/temp'), is_job=False)

## Task Code

In [0]:
import pandas as pd
import numpy as np
from datetime import date, datetime
from dateutil.relativedelta import relativedelta

# from catboost import CatBoostRegressor
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error,
)

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from pyspark.sql.window import Window
from pyspark.sql.functions import log1p, col, row_number
from pyspark.sql.functions import month as month_col
from pyspark.sql.functions import lit
from pyspark.sql.functions import min as f_min
from pyspark.sql.functions import max as f_max
from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType
import tempfile
from redshift import jdbc_to_redshift
from constants import (
    categorical,
    target,
    integer,
    numerical,
    traffic,
    add,
    SALES_FORECAST_MODEL_NAME,
)

from collections import Counter

In [0]:
boundary_query = f"""
SELECT 
    location_num,
    boundary_5,
    boundary_10,
    boundary_15,
    boundary_20,
    boundary_25,
    boundary_30,
    months_6m,
    months_12m,
    months_18m,
    months_24m,
    months_30m,
    months_36m,
    avg_monthly_sales_closest_1year,
    avg_monthly_sales_mean_1year,
    avg_monthly_sales_25th_1year,
    avg_monthly_sales_75th_1year,
    growth_rate_closest,
    growth_rate_mean,
    growth_rate_25th,
    growth_rate_75th,
    avg_monthly_sales
FROM 
    {adh.conf.default_schema_qualified}.boundary_features

"""

boundary_features = spark.sql(boundary_query)
display(boundary_features.limit(25))

location_num,boundary_5,boundary_10,boundary_15,boundary_20,boundary_25,boundary_30,months_6m,months_12m,months_18m,months_24m,months_30m,months_36m,avg_monthly_sales_closest_1year,avg_monthly_sales_mean_1year,avg_monthly_sales_25th_1year,avg_monthly_sales_75th_1year,growth_rate_closest,growth_rate_mean,growth_rate_25th,growth_rate_75th,avg_monthly_sales
00332,0,0,0,0,0,0,0,0,0,1,0,0,null,null,null,null,null,null,null,null,null
00456,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null
02053,0,0,0,0,0,0,0,0,1,2,0,0,null,null,null,null,null,null,null,null,null
02057,2,0,0,0,0,2,2,0,0,0,0,0,null,null,null,null,null,null,null,null,null
02090,0,2,0,1,0,0,1,0,1,0,0,1,null,null,null,null,null,null,null,null,null
02943,1,1,0,0,1,5,0,1,0,0,1,0,null,null,null,null,null,null,null,null,null
03098,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null
03442,0,2,3,7,1,3,0,0,1,1,0,2,505045.6558333333,322383.9778333333,265365.50625,396225.40770833334,7808.367237762246,5645.352527972034,5046.700830419586,7622.488199300704,361205.16
03511,0,2,2,3,0,0,0,0,0,0,0,0,null,null,null,null,null,null,null,null,null
03788,1,1,1,0,0,0,0,0,0,0,0,0,323204.11583333334,419040.4991666667,392306.3120833333,466958.6908333333,4976.671363636368,6160.79266899768,5406.643933566443,6752.853321678336,564980.2766666667


In [0]:
display(boundary_features.where(boundary_features.))

In [0]:
hexbin_query = f"""
SELECT 
    location_num,
    bigbox_1,
    bigbox_2,
    bigbox_3,
    competitor_1,
    competitor_2,
    competitor_3,
    highway_5,
    highway_10,
    highway_5min,
    highway_10min,
    close_highway_dist,
    close_highway_min,
    close_ramp_1,
    close_ramp_2,
    interstate_within_2min,
    median_average_daily_traffic_1,
    max_average_daily_traffic_1,
    median_average_daily_traffic_2,
    max_average_daily_traffic_2,
    median_average_daily_traffic_3,
    max_average_daily_traffic_3
FROM 
    {adh.conf.default_schema_qualified}.hexbin_level_features
"""

hexbin_features = spark.sql(hexbin_query)
display(hexbin_features.limit(25))

location_num,bigbox_1,bigbox_2,bigbox_3,competitor_1,competitor_2,competitor_3,highway_5,highway_10,highway_5min,highway_10min,close_highway_dist,close_highway_min,close_ramp_1,close_ramp_2,interstate_within_2min,median_average_daily_traffic_1,max_average_daily_traffic_1,median_average_daily_traffic_2,max_average_daily_traffic_2,median_average_daily_traffic_3,max_average_daily_traffic_3
00004,8,28,10,3,17,8,1,0,0,1,2.7721123676909474,5.187836320445173,1,0,0,31493.42857143,31860.0,8255.28571429,32463.14285714,6757.42857143,38457.0
00006,10,39,21,3,19,17,1,0,1,0,1.3698161681509344,4.945728035915581,1,0,0,3229.642857145,62792.42857143,4066.6785714275,72712.42857143,7295.2142857175,67706.28571429
00008,3,28,12,7,23,10,1,0,1,0,0.6828405037959281,2.4901080217253186,1,0,0,11343.285714285,73428.42857143,6696.07142857,66481.14285714,7092.071428575,72983.71428571
00009,11,11,2,11,3,4,1,0,1,0,1.3790702587360533,4.236081069984811,1,0,0,891.14285714,16912.71428571,7228.2857142875,51588.42857143,5815.07142857,51976.42857143
00010,3,13,2,2,12,8,1,0,0,1,2.5106936066306984,5.213194180757084,1,0,0,6204.42857143,11924.14285714,4010.1071428575005,66002.42857143,2939.5,66002.42857143
00015,7,14,9,8,10,7,0,0,0,0,100000.0,100000.0,0,0,0,8549.142857145,20930.14285714,3357.714285715,19923.0,3744.3571428575,21340.28571429
00016,0,5,0,3,3,3,1,0,1,0,0.8424522688677818,3.241838873868475,1,0,0,3402.071428575,9929.85714286,5668.892857145,66425.28571429,1564.28571429,66934.57142857
00017,3,0,6,5,0,10,0,1,0,1,5.724789254411721,9.502993070834204,0,0,0,1273.85714286,4980.71428571,4674.607142855,64319.42857143,2114.57142857,50460.28571429
00018,6,11,19,3,3,17,0,0,0,0,100000.0,100000.0,0,0,0,15355.785714285,15946.0,4798.2142857125,18393.57142857,12234.07142857,17747.57142857
0001FCB0-B2C4-4034-B078-2EA11528EB0E,3,11,5,5,11,3,0,0,0,0,100000.0,100000.0,0,0,0,8881.07142857,22295.71428571,5421.8928571425,22413.71428571,2283.0,18575.71428571


In [0]:
display(hexbin_features.limit(250))

location_num,bigbox_1,bigbox_2,bigbox_3,competitor_1,competitor_2,competitor_3,highway_5,highway_10,highway_5min,highway_10min,close_highway_dist,close_highway_min,close_ramp_1,close_ramp_2,interstate_within_2min,median_average_daily_traffic_1,max_average_daily_traffic_1,median_average_daily_traffic_2,max_average_daily_traffic_2,median_average_daily_traffic_3,max_average_daily_traffic_3
00004,8,28,10,3,17,8,1,0,0,1,2.7721123676909474,5.187836320445173,1,0,0,31493.42857143,31860.0,8255.28571429,32463.14285714,6757.42857143,38457.0
00006,10,39,21,3,19,17,1,0,1,0,1.3698161681509344,4.945728035915581,1,0,0,3229.642857145,62792.42857143,4066.6785714275,72712.42857143,7295.2142857175,67706.28571429
00008,3,28,12,7,23,10,1,0,1,0,0.6828405037959281,2.4901080217253186,1,0,0,11343.285714285,73428.42857143,6696.07142857,66481.14285714,7092.071428575,72983.71428571
00009,11,11,2,11,3,4,1,0,1,0,1.3790702587360533,4.236081069984811,1,0,0,891.14285714,16912.71428571,7228.2857142875,51588.42857143,5815.07142857,51976.42857143
00010,3,13,2,2,12,8,1,0,0,1,2.5106936066306984,5.213194180757084,1,0,0,6204.42857143,11924.14285714,4010.1071428575005,66002.42857143,2939.5,66002.42857143
00015,7,14,9,8,10,7,0,0,0,0,100000.0,100000.0,0,0,0,8549.142857145,20930.14285714,3357.714285715,19923.0,3744.3571428575,21340.28571429
00016,0,5,0,3,3,3,1,0,1,0,0.8424522688677818,3.241838873868475,1,0,0,3402.071428575,9929.85714286,5668.892857145,66425.28571429,1564.28571429,66934.57142857
00017,3,0,6,5,0,10,0,1,0,1,5.724789254411721,9.502993070834204,0,0,0,1273.85714286,4980.71428571,4674.607142855,64319.42857143,2114.57142857,50460.28571429
00018,6,11,19,3,3,17,0,0,0,0,100000.0,100000.0,0,0,0,15355.785714285,15946.0,4798.2142857125,18393.57142857,12234.07142857,17747.57142857
0001FCB0-B2C4-4034-B078-2EA11528EB0E,3,11,5,5,11,3,0,0,0,0,100000.0,100000.0,0,0,0,8881.07142857,22295.71428571,5421.8928571425,22413.71428571,2283.0,18575.71428571


In [0]:
location_attributes_query = f"""
SELECT 
    location_num,
    location_name,
    latitude,
    longitude,
    subtype,
    location_type,
    location_status,
    state,
    dma,
    submarket,
    region,
    opened_dt as open_date,
    DATEADD(YEAR, 1, opened_dt)::date AS anniversary,
    hex_string,
    ssam_flag,
    no_dt_flag,
    total_population_1,
    worker_population_1,
    daytime_population_1,
    avg_population_growth_1,
    total_households_1,
    total_households_per_sq_mile_1,
    density_classification_1,
    med_household_income_1,
    med_disposable_income_1,
    avg_disposable_income_1,
    annual_budget_expenditure_1,
    annual_budget_expenditure_index_1,
    med_home_value_1,
    avg_home_value_1,
    food_away_from_home_1,
    competitor_count_1
FROM
    {adh.conf.default_schema_qualified}.location_features
WHERE
    location_type in ('existing', 'pipeline')
"""
location_attr = spark.sql(location_attributes_query)

display(location_attr.limit(25))

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,submarket,region,open_date,anniversary,hex_string,ssam_flag,no_dt_flag,total_population_1,worker_population_1,daytime_population_1,avg_population_growth_1,total_households_1,total_households_per_sq_mile_1,density_classification_1,med_household_income_1,med_disposable_income_1,avg_disposable_income_1,annual_budget_expenditure_1,annual_budget_expenditure_index_1,med_home_value_1,avg_home_value_1,food_away_from_home_1,competitor_count_1
06980,Nestles Rd. & Los Coches Rd. FSU,36.4146998397,-121.3189742714,FSU,pipeline,future,CA,"Monterey-Salinas, CA",null,null,2029-06-07,2030-06-07,8829a9bac3fffff,0,1,6980,1879,5938,0.22,1712,60.54961391,7,76352.00,59926.00,67046.00,130371699.00,76.00,543822.00,601475.00,6020926.00,8
06989,Ogden Rd. & Hwy 550 FSU,38.45063763,-107.86579691,FSU,pipeline,future,CO,"Grand Junction-Montrose, CO",null,null,2029-06-07,2030-06-07,88269d5125fffff,0,1,1922,2933,4162,0.83,934,33.03349264,7,65669.00,52694.00,73524.00,79492036.00,85.00,432143.00,495185.00,3178496.00,14
06988,Ponca City FSU,36.73167345,-97.06672378,FSU,pipeline,future,OK,"Oklahoma City, OK",null,null,2028-08-03,2029-08-03,8826c5942dfffff,0,1,6895,5020,8099,-0.12,2825,99.91393650,7,61239.00,50765.00,66915.00,218484448.00,77.00,164099.00,209412.00,8573417.00,15
06990,Mineral Wells FSU,32.81036926,-98.07854546,FSU,pipeline,future,TX,"Dallas-Ft. Worth, TX",null,null,2028-08-03,2029-08-03,8826ca0c65fffff,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
06987,Fishermans Wharf,37.8084655,-122.41266763,FSU,pipeline,future,CA,"San Francisco-Oakland-San Jose, CA",San Francisco-Downtown,null,2029-08-03,2030-08-03,88283080ddfffff,0,1,46834,38668,66579,-0.37,24640,871.46173291,5,114359.00,92387.00,117780.00,3750307331.00,152.00,1703557.00,1622843.00,165968562.00,6
06984,Houston Downtown Inline,29.7595389,-95.36322893,FSU,pipeline,future,TX,"Houston, TX",Houston - Downtown - University Medical District - River Oaks - Heights,null,2029-08-03,2030-08-03,88446ca9b9fffff,0,1,28889,176555,165426,1.87,11535,407.96717083,6,93903.00,77189.00,105969.00,1382890759.00,120.00,408597.00,475884.00,62679625.00,32
04183,Old Hickory Blvd. (TN) FSU,36.03997,-86.767703,FSU,pipeline,future,TN,"Nashville, TN",Nashville | Brentwood - Nolensville - Antioch,Atlantic Region,2029-11-15,2030-11-15,88264d0239fffff,0,1,6762,7815,10138,1.24,3398,120.17966593,6,118915.00,100238.00,127021.00,486026742.00,143.00,581851.00,639892.00,20012462.00,6
06975,12 Mile & Mound (MI) FSU,42.50516241,-83.04589638,FSU,pipeline,future,MI,"Detroit, MI",Detroit-Sterling Heights,null,2028-09-21,2029-09-21,882ab2c2cdfffff,0,1,7256,7127,34150,0.19,3116,110.20595616,6,83151.00,67272.00,75914.00,265669051.00,85.00,230330.00,283156.00,10657332.00,16
06974,I-5 & Euclid FSU,33.83796576,-117.94190044,FSU,pipeline,future,CA,"Los Angeles, CA","Orange County, CA-Anaheim",null,2030-03-07,2031-03-07,8829a0b4adfffff,0,1,34173,12836,30039,0.18,10283,363.68672888,6,83370.00,67825.00,77083.00,893167223.00,87.00,793609.00,851976.00,40110575.00,22
06978,Radford FSU,37.13748091,-80.57027464,FSU,pipeline,future,VA,"Roanoke-Lynchburg, VA",null,Atlantic Region,2028-08-31,2029-08-31,882a8ac5a3fffff,0,1,5353,3169,7348,1.02,2404,85.02410738,7,44794.00,37591.00,48444.00,134453475.00,56.00,276103.00,302166.00,5529888.00,10


### Create final dataframe and display columns and sample of data

In [0]:
df_all = location_attr.join(boundary_features, "location_num", "inner") \
    .join(hexbin_features, "location_num", "inner") \
    .drop(boundary_features["location_num"], hexbin_features["location_num"])


In [0]:
display(df_all)

location_num,location_name,latitude,longitude,subtype,location_type,location_status,state,dma,submarket,open_date,anniversary,hex_string,ssam_flag,no_dt_flag,total_population_1,worker_population_1,daytime_population_1,avg_population_growth_1,total_households_1,total_households_per_sq_mile_1,density_classification_1,med_household_income_1,med_disposable_income_1,avg_disposable_income_1,annual_budget_expenditure_1,annual_budget_expenditure_index_1,med_home_value_1,avg_home_value_1,food_away_from_home_1,competitor_count_1,boundary_5,boundary_10,boundary_15,boundary_20,boundary_25,boundary_30,months_6m,months_12m,months_18m,months_24m,months_30m,months_36m,avg_monthly_sales_closest_1year,avg_monthly_sales_mean_1year,avg_monthly_sales_25th_1year,avg_monthly_sales_75th_1year,growth_rate_closest,growth_rate_mean,growth_rate_25th,growth_rate_75th,avg_monthly_sales,bigbox_1,bigbox_2,bigbox_3,competitor_1,competitor_2,competitor_3,highway_5,highway_10,highway_5min,highway_10min,close_highway_dist,close_highway_min,close_ramp_1,close_ramp_2,interstate_within_2min,median_average_daily_traffic_1,max_average_daily_traffic_1,median_average_daily_traffic_2,max_average_daily_traffic_2,median_average_daily_traffic_3,max_average_daily_traffic_3
06980,Nestles Rd. & Los Coches Rd. FSU,36.4146998397,-121.3189742714,FSU,pipeline,future,CA,"Monterey-Salinas, CA",null,2029-06-07,2030-06-07,8829a9bac3fffff,0,1,6980,1879,5938,0.22,1712,60.54961391,7,76352.00,59926.00,67046.00,130371699.00,76.00,543822.00,601475.00,6020926.00,8,0,0,0,0,0,2,0,0,0,0,0,1,1469290.6666666667,1469290.6666666667,1469290.6666666667,1469290.6666666667,-3841.06993006997,-3841.06993006997,-3841.06993006997,-3841.06993006997,null,2,3,4,3,6,0,0,0,0,0,100000.0,100000.0,0,0,0,5039.71428571,19236.42857143,2550.035714285,19488.71428571,1884.75,18357.57142857
06989,Ogden Rd. & Hwy 550 FSU,38.45063763,-107.86579691,FSU,pipeline,future,CO,"Grand Junction-Montrose, CO",null,2029-06-07,2030-06-07,88269d5125fffff,0,1,1922,2933,4162,0.83,934,33.03349264,7,65669.00,52694.00,73524.00,79492036.00,85.00,432143.00,495185.00,3178496.00,14,0,0,0,0,0,0,0,0,0,0,0,0,1232501.5,1232501.5,1232501.5,1232501.5,674.3566433566457,674.3566433566457,674.3566433566457,674.3566433566457,null,3,9,4,1,9,7,0,0,0,0,100000.0,100000.0,0,0,0,9530.714285715,9608.42857143,4298.678571427499,17077.28571429,1832.1785714275002,17227.0
06988,Ponca City FSU,36.73167345,-97.06672378,FSU,pipeline,future,OK,"Oklahoma City, OK",null,2028-08-03,2029-08-03,8826c5942dfffff,0,1,6895,5020,8099,-0.12,2825,99.91393650,7,61239.00,50765.00,66915.00,218484448.00,77.00,164099.00,209412.00,8573417.00,15,0,0,0,0,0,0,3,0,1,2,0,1,171857.41666666666,734450.3301282052,654637.625,881253.0416666666,1149.4160839160816,3709.9296664873714,3556.217657342677,7970.1013986014,null,9,16,2,9,6,0,0,0,0,0,100000.0,100000.0,0,0,0,5488.714285715,6215.28571429,5135.821428572501,6778.28571429,3606.357142855,7613.42857143
06990,Mineral Wells FSU,32.81036926,-98.07854546,FSU,pipeline,future,TX,"Dallas-Ft. Worth, TX",null,2028-08-03,2029-08-03,8826ca0c65fffff,0,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,4,2,1,3,5,4,228354.58333333334,803707.1582125605,666234.6458333334,939680.125,1390.3601398601377,8123.383145839678,4226.994755244767,9104.273601398607,null,0,8,5,0,8,8,0,0,0,0,100000.0,100000.0,0,0,0,8406.857142854999,8433.14285714,3424.0,10753.57142857,2121.0,11085.85714286
06987,Fishermans Wharf,37.8084655,-122.41266763,FSU,pipeline,future,CA,"San Francisco-Oakland-San Jose, CA",San Francisco-Downtown,2029-08-03,2030-08-03,88283080ddfffff,0,1,46834,38668,66579,-0.37,24640,871.46173291,5,114359.00,92387.00,117780.00,3750307331.00,152.00,1703557.00,1622843.00,165968562.00,6,0,0,0,0,0,0,1,0,2,1,3,1,1856698.8333333333,968938.036111111,704903.7708333334,1232031.875,28449.727272727345,14974.374009324027,4616.305069930105,14413.66520979024,null,0,1,2,0,5,2,0,0,0,0,100000.0,100000.0,0,0,0,1689.42857143,4116.57142857,1884.21428

In [0]:
df_all = df_all.withColumn("avg_monthly_sales_log", log1p(col("avg_monthly_sales")))
df_all = df_all.withColumn("month_of_opening", month_col(col("open_date")))
print(df_all.count())

3432


In [0]:
existing = df_all.toPandas()
print(existing.shape)
existing.to_csv('/Workspace/Users/su.bae@cfacorp.com/sales_features.csv', index=False)

(3432, 76)


In [0]:
existing = df_all.toPandas()
existing = existing[existing['location_type'] == 'existing']
print(existing.shape)
existing.to_csv('/Workspace/Users/su.bae@cfacorp.com/90_features_existing.csv', index=False)

(2866, 76)


In [0]:
future = df_all.toPandas()
future = future[future['location_type'] == 'pipeline']
print(future.shape)
future.to_csv('/Workspace/Users/su.bae@cfacorp.com/90_features_future.csv', index=False)

(566, 76)


In [0]:
for field in df_all.schema.fields:
    if isinstance(field.dataType, DecimalType):
        df_all = df_all.withColumn(field.name, col(field.name).cast("double"))

### Document which version of catboost is being used

In [0]:
%sh
pip --disable-pip-version-check list | grep catboost

### Convert to Pandas for model training

In [0]:
EXPERIMENT_NAME = f"{adh.conf.default_schema}_{adh.params.task_key}_{adh.params.run_id}"
FULL_EXPERIMENT_NAME = f"/Shared/{EXPERIMENT_NAME}"
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"{FULL_EXPERIMENT_NAME=} {now=}")

Generate test / train split and enforce assertion that there is no training data newer than the test data

In [0]:
display(df_all)

In [0]:
full_set = df_all.na.drop()

window = Window.orderBy("open_date")
full_set_with_rownum = full_set.withColumn("row_num", row_number().over(window))

total_count = full_set_with_rownum.count()
print(total_count)
split_index = int(total_count * 0.9)

df_train_split = full_set_with_rownum.where(col("row_num") <= split_index).drop("row_num")
df_test_split = full_set_with_rownum.where(col("row_num") > split_index).drop("row_num")

df = df_all.where(df_all.location_num.cast('string').startswith('0')).toPandas()
print(df.shape)


In [0]:
def median_ape(y_true, y_pred):
    return np.median(np.abs((y_true - y_pred) / y_true))

In [0]:
for year in [2023, 2024, 2025]:
    for month in [1]:
        c, n = str(datetime.strftime(date(year,month,1), '%Y-%m-%d'))[:10], str(datetime.strftime(date(year,month,1) + relativedelta(months=+12), '%Y-%m-%d'))[:10]
        train, test = df[df['anniversary'] < pd.to_datetime(c)], df[(df['anniversary'] >= pd.to_datetime(c)) & (df['anniversary'] < pd.to_datetime(n))]
        print(c, n, train.shape, test.shape)

        train, test = train[train[target].notna()], test[test[target].notna()]
        print(c, n, train.shape, test.shape)
        x_train, y_train = train[add].values.reshape(-1, len(add)), train[target].values.reshape(-1,1)
        x_test, y_test = test[add].values.reshape(-1, len(add)), test[target].values.reshape(-1,1)
        categorical_features_indices = np.where(train[add].dtypes == object)[0]
        #for name, model in pipelines.items():
        model = CatBoostRegressor(iterations=200, rsm = 0.9, score_function = "L2", depth=6, learning_rate=0.05, eval_metric="MAPE", verbose=False)
        model.fit(x_train, y_train, cat_features=categorical_features_indices)#, eval_set=(X_validation, y_validation))

        # validate
        y_1, y_2 = np.expm1(model.predict(x_train)), np.expm1(model.predict(x_test))
        y_train, y_test = np.expm1(y_train), np.expm1(y_test)
        
        output = []
        for i, j in zip(y_test, y_2):
            output += [round(abs(i[0] - j)/i[0] * 100, 1)//10]
        print(Counter(output), median_ape(y_test, y_2))
        print('cbr', mean_absolute_error(y_train, y_1), mean_absolute_percentage_error(y_train, y_1), mean_absolute_error(y_test, y_2), mean_absolute_percentage_error(y_test, y_2))

In [0]:
test_min_open_date = df_test_split.select(f_min(df_test_split.open_date)).collect()[0][
    0
]
test_max_open_date = df_test_split.select(f_max(df_test_split.open_date)).collect()[0][
    0
]
train_min_open_date = df_train_split.select(f_min(df_train_split.open_date)).collect()[
    0
][0]
train_max_open_date = df_train_split.select(f_max(df_train_split.open_date)).collect()[
    0
][0]
assert (
    train_max_open_date <= test_min_open_date
)  # NOTE: this is a data validation check as no values in the test set should be before the train set as well as no values in the train set should be after the test set
print(train_min_open_date, train_max_open_date, test_min_open_date, test_max_open_date)


### Feature distributions for train and test sets for enhanced debugging

In [0]:
distributions = {}
for column in integer + numerical + traffic:
    quantiles = df_train_split.approxQuantile(column, [i / 4 for i in range(5)], 0.01)
    histogram = [(quantiles[i], quantiles[i + 1]) for i in range(len(quantiles) - 1)]
    distributions[column] = histogram

distribution_df = pd.DataFrame.from_dict(
    distributions, orient="index", columns=["Q1-Q2", "Q2-Q3", "Q3-Q4", "Q4-Q5"]
).assign(column=list(distributions.keys()))
distribution_df = distribution_df[["column", "Q1-Q2", "Q2-Q3", "Q3-Q4", "Q4-Q5"]]
display(distribution_df)

In [0]:
test_distributions = {}
for column in integer + numerical + traffic:
    quantiles = df_test_split.approxQuantile(column, [i / 4 for i in range(5)], 0.01)
    histogram = [(quantiles[i], quantiles[i + 1]) for i in range(len(quantiles) - 1)]
    test_distributions[column] = histogram

test_distribution_df = pd.DataFrame.from_dict(
    test_distributions, orient="index", columns=["Q1-Q2", "Q2-Q3", "Q3-Q4", "Q4-Q5"]
).assign(column=list(test_distributions.keys()))
test_distribution_df = test_distribution_df[
    ["column", "Q1-Q2", "Q2-Q3", "Q3-Q4", "Q4-Q5"]
]
display(test_distribution_df)

In [0]:
mlflow_experiment = mlflow.set_experiment(experiment_name=FULL_EXPERIMENT_NAME)


def train_sales_forecast_model(train_spdf: DataFrame, test_spdf: DataFrame):
    with mlflow.start_run(run_name=f"sales-forecast-{now}-{adh.params.run_id}"):
        train_spdf, test_spdf = train_spdf.na.drop(), test_spdf.na.drop()
        whole = train_spdf.union(test_spdf)
        whole_size, train_size, test_size = (
            whole.count(),
            train_spdf.count(),
            test_spdf.count(),
        )
        print(f"{whole_size=} {train_size=}, {test_size=} going into model training")
        whole, train, test = (
            whole.toPandas(),
            train_spdf.toPandas(),
            test_spdf.toPandas(),
        )

        whole[integer], train[integer], test[integer] = (
            whole[integer].astype(int),
            train[integer].astype(int),
            test[integer].astype(int),
        )
        whole[categorical], train[categorical], test[categorical] = (
            whole[categorical].astype(str),
            train[categorical].astype(str),
            test[categorical].astype(str),
        )
        whole[numerical], train[numerical], test[numerical] = (
            whole[numerical].astype(float),
            train[numerical].astype(float),
            test[numerical].astype(float),
        )
        whole[traffic], train[traffic], test[traffic] = (
            whole[traffic].astype(float),
            train[traffic].astype(float),
            test[traffic].astype(float),
        )
        add = categorical + numerical + traffic
        x_whole, y_whole = whole[add].values, whole[target].values.reshape(-1, 1)
        x_train, y_train = train[add].values, train[target].values.reshape(-1, 1)
        x_test, y_test = test[add].values, test[target].values.reshape(-1, 1)

        categorical_features_indices = np.where(train[add].dtypes == object)[0]
        print(f"{categorical_features_indices=}")

        model = CatBoostRegressor(
            iterations=200,
            rsm=0.9,
            score_function="L2",
            depth=6,
            learning_rate=0.05,
            eval_metric="MAPE",
            verbose=False,
        )
        metrics_model = model.fit(
            x_train, y_train, cat_features=categorical_features_indices
        )

        # validate
        predict_train, predict_test = np.expm1(
            metrics_model.predict(x_train)
        ), np.expm1(metrics_model.predict(x_test))
        y_train, y_test = np.expm1(y_train).flatten(), np.expm1(y_test).flatten()
        combined_train = pd.DataFrame(x_train, columns=add).assign(
            actual=y_train.flatten(), predicted=predict_train.flatten()
        )
        combined_test = pd.DataFrame(x_test, columns=add).assign(
            actual=y_test.flatten(), predicted=predict_test.flatten()
        )

        train_with_predictions = train.assign(predicted=predict_train.flatten())
        test_with_predictions = test.assign(predicted=predict_test.flatten())

        # TODO: investigate breakdown by year
        mae_train, mape_train, r2_train, rmse_train = (
            mean_absolute_error(y_train, predict_train),
            mean_absolute_percentage_error(y_train, predict_train),
            r2_score(y_train, predict_train),
            mean_squared_error(y_train, predict_train, squared=False),
        )
        mae_test, mape_test, r2_test, rmse_test = (
            mean_absolute_error(y_test, predict_test),
            mean_absolute_percentage_error(y_test, predict_test),
            r2_score(y_test, predict_test),
            mean_squared_error(y_test, predict_test, squared=False),
        )
        median_ape_train, median_ape_test = median_ape(
            y_train, predict_train
        ), median_ape(y_test, predict_test)

        print(
            f"{mae_train=}, {mape_train=}, {r2_train=}, {rmse_train=}, {median_ape_train=}, {mae_test=}, {mape_test=}, {r2_test=}, {rmse_test=}, {median_ape_test=}"
        )

        metrics = {
            "MAPE_train": mape_train,
            "MAPE_test": mape_test,
            "MAE_train": mae_train,
            "MAE_test": mae_test,
            "R2_train": r2_train,
            "R2_test": r2_test,
            "RMSE_train": rmse_train,
            "RMSE_test": rmse_test,
            "train_data_size": train_size,
            "test_data_size": test_size,
            "median_ape_train": median_ape_train,
            "median_ape_test": median_ape_test,
        }
        mlflow.log_metrics(metrics)

        trained_model = model.fit(
            x_whole, y_whole, cat_features=categorical_features_indices
        )  # retrain model with whole dataset
        pandas_schema = Schema(
            [
                ColSpec(
                    type="double"
                    if str(dtype) == "float64"
                    else "long"
                    if str(dtype) == "int64"
                    else "string",
                    name=col_name,
                )
                for col_name, dtype in whole[add].dtypes.items()
            ]
        )
        output_schema = Schema([ColSpec("double")])
        mlflow.catboost.log_model(
            trained_model,
            artifact_path=SALES_FORECAST_MODEL_NAME,
            registered_model_name=SALES_FORECAST_MODEL_NAME,
            signature=ModelSignature(inputs=pandas_schema, outputs=output_schema),
            input_example=whole[add].iloc[[0]],
        )

        mlflow.log_input(
            mlflow.data.load_delta(
                table_name=f"{adh.conf.default_schema_qualified}.hexbin_level_features"
            ),
            context="training",
        )
        mlflow.log_input(
            mlflow.data.load_delta(
                table_name=f"{adh.conf.default_schema_qualified}.boundary_features"
            ),
            context="training",
        )
        # tmp_dir = "/tmp/sales_forecast"
        # os.makedirs(tmp_dir, exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp_dir:
            combined_train.to_csv(f"{tmp_dir}/train_features.csv")
            combined_test.to_csv(f"{tmp_dir}/test_features.csv")
            train_with_predictions.to_csv(f"{tmp_dir}/train_with_predictions.csv")
            test_with_predictions.to_csv(f"{tmp_dir}/test_with_predictions.csv")
            mlflow.log_artifacts(tmp_dir, artifact_path="data")

        # assert len(combined_train[(combined_train['predicted']*12) < 3000000]) == 0, f"all forecasts should be atleast $3,000,000 annually {combined_train[(combined_train['predicted']*12) < 3000000]=}"
        # assert len(combined_test[(combined_test['predicted']*12) < 3000000]) == 0, f"all forecasts should be atleast $3,000,000 annually {combined_test[(combined_test['predicted']*12) < 3000000]=}"


# Testing inferring the schema

In [0]:
train_sales_forecast_model(df_train_split, df_test_split)

In [0]:
client = MlflowClient()

In [0]:
print((mlflow.last_active_run().info))


In [0]:
model_name = f"{adh.conf.default_schema_qualified}.mom_sales_forecast"
run_id = mlflow.last_active_run().info.run_id
print(run_id)

# Search for model versions by model name
model_versions = client.search_model_versions(f"name = '{model_name}'")

# Filter the results by run_id
filtered_model_versions = [mv.version for mv in model_versions if mv.run_id == run_id]
assert (
    len(filtered_model_versions) == 1
), f"{run_id=} {model_name=} {filtered_model_versions=}"


In [0]:
print(model_name)
print(run_id)
print(filtered_model_versions)

In [0]:

metric_df = spark.createDataFrame(
    mlflow.last_active_run().data.metrics.items(), ["metric_type", "metric"]
)
metric_df = metric_df.withColumn(
    "run_name", lit(mlflow.last_active_run().info.run_name)
)
display(metric_df)

In [0]:
if adh.conf.default_schema == "mom_capacity_ml_cc" and adh.params.environment in (
    "dev",
    "qa",
    "prod",
):
    client.set_model_version_tag(
        model_name, filtered_model_versions[-1], "model_status", "in_validation"
    )
    jdbc_to_redshift(
        metric_df,
        f'market_optimization.sales_forecast_metrics{ "" if adh.params.environment == "prod" else f"_{adh.params.environment}"}',
    )
else:
    client.set_model_version_tag(
        model_name, filtered_model_versions[-1], "model_status", "local"
    )
    print("Skipping metrics upload to RedShift")